In [7]:
import re
import random
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị đang dùng:", device)

Thiết bị đang dùng: cpu


In [8]:
dataset_path = Path("truyen_kieu_data.txt")
raw_text = dataset_path.read_text(encoding="utf-8")

print(f"Đã load {dataset_path} thành công.")
print(f"Số ký tự: {len(raw_text):,}")
print("Mẫu 300 ký tự đầu:")
print(raw_text[:300])

Đã load truyen_kieu_data.txt thành công.
Số ký tự: 108,741
Mẫu 300 ký tự đầu:
1..Trăm năm trong cõi người ta,
2..Chữ tài chữ mệnh khéo là ghét nhau.
3..Trải qua một cuộc bể dâu,
4..Những điều trông thấy mà đau đớn lòng.
5.. Lạ gì bỉ sắc tư phong,
6..Trời xanh quen thói má hồng đánh ghen.
7..Cảo thơm lần giở trước đèn,
8..Phong tình có lục còn truyền sử xanh.
9,,Rằng năm Gia T


In [9]:
def normalize_text_for_ngram(text: str) -> str:
    text = text.lower()
    text = re.sub(r"(?m)^\s*\d+[.,]{1,2}\s*", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_words_and_punctuation(text: str):
    return re.findall(r"\w+|[.,!?;:]", text, flags=re.UNICODE)


clean_text = normalize_text_for_ngram(raw_text)
all_tokens = tokenize_words_and_punctuation(clean_text)

split_index = int(0.8 * len(all_tokens))
train_tokens = all_tokens[:split_index]
test_tokens = all_tokens[split_index:]

minimum_word_frequency = 2
train_word_frequency = Counter(train_tokens)


def map_tokens_for_language_model(tokens, train_word_frequency, minimum_word_frequency):
    mapped_tokens = []
    for token in tokens:
        if train_word_frequency.get(token, 0) >= minimum_word_frequency:
            mapped_tokens.append(token)
        else:
            mapped_tokens.append("<unk>")
    return mapped_tokens


train_tokens_for_model = map_tokens_for_language_model(
    train_tokens,
    train_word_frequency,
    minimum_word_frequency
)

test_tokens_for_model = map_tokens_for_language_model(
    test_tokens,
    train_word_frequency,
    minimum_word_frequency
)

unknown_count_in_test = sum(token == "<unk>" for token in test_tokens_for_model)

print(f"Tổng số token: {len(all_tokens):,}")
print(f"Train token: {len(train_tokens):,} ({len(train_tokens)/len(all_tokens):.2%})")
print(f"Test token: {len(test_tokens):,} ({len(test_tokens)/len(all_tokens):.2%})")
print(f"Ngưỡng từ hiếm: >= {minimum_word_frequency}")
print(f"Tỷ lệ <unk> trong test: {unknown_count_in_test/len(test_tokens_for_model):.2%}")
print("20 token đầu train (sau map):", train_tokens_for_model[:20])

Tổng số token: 26,519
Train token: 21,215 (80.00%)
Test token: 5,304 (20.00%)
Ngưỡng từ hiếm: >= 2
Tỷ lệ <unk> trong test: 5.67%
20 token đầu train (sau map): ['trăm', 'năm', 'trong', 'cõi', 'người', 'ta', ',', 'chữ', 'tài', 'chữ', 'mệnh', 'khéo', 'là', 'ghét', 'nhau', '.', 'trải', 'qua', 'một', 'cuộc']


In [10]:
def build_vocabulary(train_tokens):
    unique_words = sorted(set(train_tokens))
    if "<unk>" not in unique_words:
        unique_words = ["<unk>"] + unique_words
    return unique_words


class NGramTextDataset(Dataset):
    def __init__(self, tokens, context_size, word_to_index):
        self.samples = []
        unknown_index = word_to_index["<unk>"]

        for token_position in range(context_size, len(tokens)):
            context_words = tokens[token_position - context_size:token_position]
            target_word = tokens[token_position]

            context_indices = [word_to_index.get(word, unknown_index) for word in context_words]
            target_index = word_to_index.get(target_word, unknown_index)
            self.samples.append((context_indices, target_index))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        context_indices, target_index = self.samples[index]
        return (
            torch.tensor(context_indices, dtype=torch.long),
            torch.tensor(target_index, dtype=torch.long)
        )


class NGramLanguageModel(nn.Module):
    def __init__(self, vocabulary_size, context_size, embedding_dimension=128, hidden_dimension=256):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dimension)
        self.hidden_layer = nn.Linear(context_size * embedding_dimension, hidden_dimension)
        self.output_layer = nn.Linear(hidden_dimension, vocabulary_size)
        self.activation = nn.ReLU()

    def forward(self, context_indices):
        embedded = self.embedding(context_indices)
        flattened = embedded.view(embedded.size(0), -1)
        hidden_state = self.activation(self.hidden_layer(flattened))
        logits = self.output_layer(hidden_state)
        return logits


class NGramTrainer:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.loss_function = nn.CrossEntropyLoss()

    def train(self, train_loader, number_of_epochs=12, learning_rate=0.002):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        training_losses = []

        for epoch in range(number_of_epochs):
            self.model.train()
            total_loss = 0.0

            for context_batch, target_batch in train_loader:
                context_batch = context_batch.to(self.device)
                target_batch = target_batch.to(self.device)

                optimizer.zero_grad()
                logits = self.model(context_batch)
                loss = self.loss_function(logits, target_batch)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            average_loss = total_loss / len(train_loader)
            training_losses.append(average_loss)
            print(f"Epoch {epoch + 1:02d}/{number_of_epochs} - Train Loss: {average_loss:.4f}")

        return training_losses

    def calculate_perplexity(self, data_loader):
        self.model.eval()
        total_loss = 0.0

        with torch.no_grad():
            for context_batch, target_batch in data_loader:
                context_batch = context_batch.to(self.device)
                target_batch = target_batch.to(self.device)
                logits = self.model(context_batch)
                loss = self.loss_function(logits, target_batch)
                total_loss += loss.item()

        average_loss = total_loss / len(data_loader)
        perplexity = torch.exp(torch.tensor(average_loss)).item()
        return perplexity

    def generate_sentence(
        self,
        seed_words,
        word_to_index,
        index_to_word,
        context_size,
        max_new_tokens=30,
        temperature=1.0,
        stop_tokens=None
    ):
        if stop_tokens is None:
            stop_tokens = {".", "!", "?"}

        self.model.eval()

        if isinstance(seed_words, str):
            generated_words = seed_words.lower().split()
        else:
            generated_words = [word.lower() for word in seed_words]

        unknown_index = word_to_index["<unk>"]

        with torch.no_grad():
            for _ in range(max_new_tokens):
                current_context_words = generated_words[-context_size:]
                if len(current_context_words) < context_size:
                    current_context_words = ["<unk>"] * (context_size - len(current_context_words)) + current_context_words

                current_context_indices = [word_to_index.get(word, unknown_index) for word in current_context_words]
                context_tensor = torch.tensor([current_context_indices], dtype=torch.long, device=self.device)

                logits = self.model(context_tensor).squeeze(0)
                scaled_logits = logits / temperature
                probabilities = torch.softmax(scaled_logits, dim=0)

                probabilities[unknown_index] = 0.0
                probability_sum = probabilities.sum()
                if probability_sum > 0:
                    probabilities = probabilities / probability_sum
                else:
                    probabilities = torch.softmax(scaled_logits, dim=0)

                next_word_index = torch.multinomial(probabilities, num_samples=1).item()
                next_word = index_to_word[next_word_index]

                generated_words.append(next_word)

                if next_word in stop_tokens:
                    break

        generated_sentence = " ".join(generated_words)
        generated_sentence = re.sub(r"\s+([.,!?;:])", r"\1", generated_sentence)
        return generated_sentence

In [13]:
n_gram_size = 2
context_size = n_gram_size - 1

vocabulary = build_vocabulary(train_tokens_for_model)
word_to_index = {word: index for index, word in enumerate(vocabulary)}
index_to_word = {index: word for word, index in word_to_index.items()}

train_dataset = NGramTextDataset(train_tokens_for_model, context_size, word_to_index)
test_dataset = NGramTextDataset(test_tokens_for_model, context_size, word_to_index)

batch_size = 512
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = NGramLanguageModel(
    vocabulary_size=len(vocabulary),
    context_size=context_size,
    embedding_dimension=128,
    hidden_dimension=256
).to(device)

trainer = NGramTrainer(model=model, device=device)

print(f"N-gram size: {n_gram_size}")
print(f"Vocabulary size: {len(vocabulary):,}")
print(f"Số mẫu train: {len(train_dataset):,}")
print(f"Số mẫu test: {len(test_dataset):,}")

training_losses = trainer.train(
    train_loader=train_loader,
    number_of_epochs=12,
    learning_rate=0.002
)

N-gram size: 2
Vocabulary size: 1,597
Số mẫu train: 21,214
Số mẫu test: 5,303
Epoch 01/12 - Train Loss: 6.4568
Epoch 02/12 - Train Loss: 5.6541
Epoch 03/12 - Train Loss: 5.1909
Epoch 04/12 - Train Loss: 4.7363
Epoch 05/12 - Train Loss: 4.3762
Epoch 06/12 - Train Loss: 4.1349
Epoch 07/12 - Train Loss: 3.9712
Epoch 08/12 - Train Loss: 3.8647
Epoch 09/12 - Train Loss: 3.7855
Epoch 10/12 - Train Loss: 3.7195
Epoch 11/12 - Train Loss: 3.6701
Epoch 12/12 - Train Loss: 3.6343


In [14]:
test_perplexity = trainer.calculate_perplexity(test_loader)
print(f"Perplexity trên tập test: {test_perplexity:.4f}")

seed_examples = [
    ["trăm", "năm"],
    ["người", "đâu"],
    ["một", "đời"],
    ["chữ", "tài"]
]

for seed_words in seed_examples:
    generated_sentence = trainer.generate_sentence(
        seed_words=seed_words,
        word_to_index=word_to_index,
        index_to_word=index_to_word,
        context_size=context_size,
        max_new_tokens=28,
        temperature=0.9
    )
    print(f"Seed {seed_words} -> {generated_sentence}")

Perplexity trên tập test: 877.4634
Seed ['trăm', 'năm'] -> trăm năm sau thì thôi lại xót cho qua: người quốc sắc nước xa nô kia nỗi mình: con bướm lả han chào, một dây oan vỡ
Seed ['người', 'đâu'] -> người đâu vội truyền sửa bộ tháng, vai về già sư, mà đau.
Seed ['một', 'đời'] -> một đời bỏ cho những ngày xuân tàng.
Seed ['chữ', 'tài'] -> chữ tài, giữa vòng.
